In [0]:
%pip install -q "psycopg[binary]>=3.1.0" "databricks-sdk>=0.118.0" databricks-langchain "langgraph>=0.3,<0.4" databricks-ai-search

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


# FGAC hardening (Step 5 follow-up), 2026-10-21
Workspace **fevm-workshop-guy**, Lakebase project `fe-bar-fraud-analytics` / `production` / `fe_bar_db`, UC catalog `fe_bar_lakebase`.

Each item prints its **CHECK** state, the **FIX** applied (or *no change needed*), and its **VERIFY** result. The cells are idempotent, so re-running them shows the current state. Where a fix was applied, the pre-fix evidence (captured in the same session before the fix) is quoted from `logs/fgac_hardening_2026-10-21.md`.

* SQL changes: `lakebase_fgac_hardening.sql` (idempotent), folded into `lakebase_fgac.sql` and `lakebase_cases_ddl.sql`
* Synced tables `fraud.*` are never altered.

In [0]:
import sys, json, hashlib, time
import psycopg, psycopg.sql as psql
from databricks.sdk import WorkspaceClient
sys.path.insert(0, "/Workspace/Users/alex.owen@databricks.com/FE-BAR-2026-repo")

w = WorkspaceClient()
ME = w.current_user.me().user_name
EP = w.postgres.get_endpoint(name="projects/fe-bar-fraud-analytics/branches/production/endpoints/primary")

def pg(autocommit=True):
    tok = w.postgres.generate_database_credential(endpoint=EP.name).token
    c = psycopg.connect(host=EP.status.hosts.host, dbname="fe_bar_db", user=ME, password=tok, sslmode="require")
    c.autocommit = autocommit
    return c

def one(cur, sql, params=()):
    cur.execute(sql, params); return cur.fetchone()

print("login:", ME, "| host:", EP.status.hosts.host)

login: alex.owen@databricks.com | host: ep-aged-sky-d2afekn0.database.us-east-1.cloud.databricks.com


In [0]:
from lakebase_cases import LakebaseCases
c = pg(); cur = c.cursor()
print("CHECK")
print("  role attrs (session_user, rolsuper, rolbypassrls):", one(cur, "SELECT rolname, rolsuper, rolbypassrls FROM pg_roles WHERE rolname = session_user"))
for t in ["cases.fraud_cases", "cases.case_actions"]:
    print(f"  {t}: (relrowsecurity, relforcerowsecurity) =", one(cur, "SELECT relrowsecurity, relforcerowsecurity FROM pg_class WHERE oid = %s::regclass", (t,)))
n_cases = one(cur, "SELECT count(*) FROM cases.fraud_cases")[0]
n_actions = one(cur, "SELECT count(*) FROM cases.case_actions")[0]
print(f"  visible with no SET ROLE: fraud_cases={n_cases}, case_actions={n_actions}")
test_txn = one(cur, """SELECT txn_id FROM fraud.txn_features WHERE txn_id <> 'TXN-0234209' AND amount <= 10000
                       AND txn_id NOT IN (SELECT txn_id FROM cases.fraud_cases WHERE status <> 'closed')
                       ORDER BY txn_id DESC LIMIT 1""")[0]
c.close()
lc = LakebaseCases()  # no pg_role
case_id = lc.create_fraud_case(test_txn, "review", "analyst_review", "hardening item 1 round trip", "fgac_hardening_item1")
lc.update_case_status(case_id, "closed", "fgac_hardening_item1")
case = lc.get_case(case_id); lc.close()
ok = n_cases > 0 and case["status"] == "closed"
print(f"  round trip on {test_txn}: case {case_id} -> status={case['status']}, actions={len(case['actions'])}, db_inserted_by={case['db_inserted_by']}")
print("FIX:", "no change needed (rolbypassrls=true, so FORCE RLS does not lock the admin out; admin policy left commented in lakebase_fgac_hardening.sql)" if ok else "NEEDED: add admin_all_* policies")
print("VERIFY:", "PASS" if ok else "FAIL", "| lakebase_write_path_validation.ipynb re-run end to end (see that notebook)")

CHECK
  role attrs (session_user, rolsuper, rolbypassrls): ('alex.owen@databricks.com', False, True)
  cases.fraud_cases: (relrowsecurity, relforcerowsecurity) = (True, True)
  cases.case_actions: (relrowsecurity, relforcerowsecurity) = (True, True)
  visible with no SET ROLE: fraud_cases=224, case_actions=448
  round trip on TXN-0500000: case 7c4f24ab-2c7b-4638-aefd-1f242a8c2bef -> status=closed, actions=2, db_inserted_by=alex.owen@databricks.com
FIX: no change needed (rolbypassrls=true, so FORCE RLS does not lock the admin out; admin policy left commented in lakebase_fgac_hardening.sql)
VERIFY: PASS | lakebase_write_path_validation.ipynb re-run end to end (see that notebook)


In [0]:
c = pg(); cur = c.cursor(); all_ok = True
print("CHECK / VERIFY")
for role in ["fraud_pii", "fraud_analyst", "fraud_lead", "fraud_triage_agent"]:
    member = one(cur, "SELECT pg_has_role(session_user, %s, 'MEMBER')", (role,))[0]
    try:
        cur.execute(psql.SQL("SET ROLE {}").format(psql.Identifier(role)))
        cu = one(cur, "SELECT current_user")[0]; cur.execute("RESET ROLE"); res = f"OK (current_user={cu})"
    except Exception as e:
        all_ok = False; res = f"FAIL: {e}"
    print(f"  {role:<20} pg_has_role MEMBER={member!s:<5}  SET ROLE/RESET ROLE: {res}")
c.close()
print("FIX:", "no change needed (grants already in lakebase_fgac.sql: GRANT <role> TO CURRENT_USER)" if all_ok else "NEEDED: GRANT <role> TO my login")
print("NOTE: the fraud_pii membership that alex.owen holds for testing is in addition to the real UC pii membership; pii_role_sync.py (Item 8) maps it.")
print("VERIFY:", "PASS" if all_ok else "FAIL", "| lakebase_fgac_validation.ipynb re-run end to end (see that notebook)")

CHECK / VERIFY
  fraud_pii            pg_has_role MEMBER=True   SET ROLE/RESET ROLE: OK (current_user=fraud_pii)
  fraud_analyst        pg_has_role MEMBER=True   SET ROLE/RESET ROLE: OK (current_user=fraud_analyst)
  fraud_lead           pg_has_role MEMBER=True   SET ROLE/RESET ROLE: OK (current_user=fraud_lead)
  fraud_triage_agent   pg_has_role MEMBER=True   SET ROLE/RESET ROLE: OK (current_user=fraud_triage_agent)
FIX: no change needed (grants already in lakebase_fgac.sql: GRANT <role> TO CURRENT_USER)
NOTE: the fraud_pii membership that alex.owen holds for testing is in addition to the real UC pii membership; pii_role_sync.py (Item 8) maps it.
VERIFY: PASS | lakebase_fgac_validation.ipynb re-run end to end (see that notebook)


In [0]:
# The job was run once in this session (run 1101647585612679). This cell reports that run and
# compares the current Postgres state with the 'before' snapshot recorded just before the run.
RUN_ID = 1101647585612679
BEFORE = {"oids": {"customer_risk_profile": 24743, "txn_features": 24712, "txn_risk_current": 73749},
          "def_md5": {"customer_risk_profile": "3034e6f23e25e8d1796fdbae69eea51e", "txn_features": "9f4c89e32a3d1fa4e6070ef484e64f95", "txn_risk_current": "e95b38e75af957ac8ee2e07fa83964ed"}}
r = w.jobs.get_run(RUN_ID)
print(f"TEST  job fe_bar_gold_to_lakebase run {RUN_ID}: {r.state.life_cycle_state.value} / {r.state.result_state.value}")
for t in sorted(r.tasks, key=lambda t: t.start_time):
    print(f"  task {t.task_key:<24} {t.state.result_state.value:<8} run_id={t.run_id}")
c = pg(); cur = c.cursor(); ok = r.state.result_state.value == "SUCCESS"
print("VERIFY synced table parent OIDs (before -> now)")
for tbl, oid in BEFORE["oids"].items():
    now = one(cur, "SELECT %s::regclass::oid", (f"fraud.{tbl}",))[0]; ok &= now == oid
    print(f"  fraud.{tbl:<22} {oid} -> {now}  {'same' if now == oid else 'RECREATED'}")
print("VERIFY fraud_gov views (owner, definition unchanged)")
for v, md5 in BEFORE["def_md5"].items():
    owner, d = one(cur, "SELECT viewowner, definition FROM pg_views WHERE schemaname='fraud_gov' AND viewname=%s", (v,))
    same = hashlib.md5(d.encode()).hexdigest() == md5; ok &= same
    print(f"  fraud_gov.{v:<22} owner={owner} same_definition={same}")
print("VERIFY SELECT counts per role")
for role in ["fraud_analyst", "fraud_lead"]:
    cur.execute(psql.SQL("SET ROLE {}").format(psql.Identifier(role)))
    cnts = {v: one(cur, psql.SQL("SELECT count(*) FROM fraud_gov.{}").format(psql.Identifier(v)))[0] for v in BEFORE["def_md5"]}
    cur.execute("RESET ROLE"); print(f"  {role:<14} {cnts}")
print("VERIFY UC vs Postgres row counts")
for pgt, uct in [("txn_features", "txn_features_gold"), ("customer_risk_profile", "customer_risk_profile_gold"), ("txn_risk_current", "scored_transactions")]:
    u = spark.sql(f"SELECT count(*) c FROM workshop_guy_catalog.fe_bar_a0.{uct}").first().c
    p = one(cur, psql.SQL("SELECT count(*) FROM fraud.{}").format(psql.Identifier(pgt)))[0]; ok &= u == p
    print(f"  {uct:<28} {u:>8,}  vs fraud.{pgt:<22} {p:>8,}  match={u == p}")
c.close()
print("FIX:", "no change needed (job succeeded, parents not recreated, views intact)" if ok else "STOP and report (do not alter synced tables)")

TEST  job fe_bar_gold_to_lakebase run 1101647585612679: TERMINATED / SUCCESS
  task gold_pipeline_refresh    SUCCESS  run_id=490435646942231
  task batch_scoring            SUCCESS  run_id=348378236508614
  task lakebase_sync_refresh    SUCCESS  run_id=742998301546873
VERIFY synced table parent OIDs (before -> now)
  fraud.customer_risk_profile  24743 -> 24743  same
  fraud.txn_features           24712 -> 24712  same
  fraud.txn_risk_current       73749 -> 73749  same
VERIFY fraud_gov views (owner, definition unchanged)
  fraud_gov.customer_risk_profile  owner=alex.owen@databricks.com same_definition=True
  fraud_gov.txn_features           owner=alex.owen@databricks.com same_definition=True
  fraud_gov.txn_risk_current       owner=alex.owen@databricks.com same_definition=True
VERIFY SELECT counts per role
  fraud_analyst  {'customer_risk_profile': 9289, 'txn_features': 463881, 'txn_risk_current': 463881}
  fraud_lead     {'customer_risk_profile': 9289, 'txn_features': 464632, 'txn_risk

## Items 4 and 6 (documentation only, no runtime change)
* **Item 4:** section *Alternative considered: filtered-view sync* in `logs/agent_governance_2026-10-21.md`, with docs links.
* **Item 6:** Part A.5 retitled *Before picture: UC FGAC is not propagated to synced tables (documented behavior)* in `lakebase_fgac_validation.ipynb` and `logs/lakebase_fgac_2026-10-21.md`, with the docs link and a sentence on what fraud_gov changes. The numbers are unchanged.

In [0]:
import re
print("CHECK  objects read by the UC functions (parsed from their definitions)")
for f in ["get_transaction_risk", "explain_top_signals", "get_customer_history"]:
    row = spark.sql(f"""SELECT routine_definition, comment FROM workshop_guy_catalog.information_schema.routines
                        WHERE routine_schema='fe_bar_a0' AND routine_name='{f}'""").first()
    objs = sorted(set(re.findall(r"(fe_bar_lakebase\.\w+\.\w+|workshop_guy_catalog\.fe_bar_a0\.\w+)", row.routine_definition)))
    print(f"  {f:<22} reads {objs}")
    print(f"  {'':<22} comment: {row.comment[:70]}...")
print("\nCHECK  UC path")
print("  UC current_user:", spark.sql("SELECT current_user()").first()[0], "| is_account_group_member('pii'):", spark.sql("SELECT is_account_group_member('pii')").first()[0])
for t in ["fraud_gov.txn_features", "fraud.txn_features", "cases.fraud_cases"]:
    r = spark.sql(f"SELECT count(*) n FROM fe_bar_lakebase.{t}").first().n
    print(f"  fe_bar_lakebase.{t:<24} rows via UC = {r:,}")
r = spark.sql("SELECT count_if(amount > 10000) hi, count(amt_to_monthly_spend) nn FROM fe_bar_lakebase.fraud_gov.txn_features").first()
print(f"  fraud_gov via UC as a pii member: high-amount rows={r.hi}, non-null amt_to_monthly_spend={r.nn}  -> filtered for everyone, not identity-aware")
print("  Non-pii identity on the UC path: NOT TESTABLE here (the only human principal available, alex.owen, is in pii).")
print("\nFIX  rule 'UC or Postgres, never neither': the UC path cannot show per-identity rules, so the three read tools")
print("     are now agent-side Python tools in fraud_triage_agent.py reading fraud_gov.* as fraud_triage_agent.")
print("     The UC functions remain in agent_tools.sql, marked DEPRECATED in their COMMENT, and are not registered with the agent.")
import fraud_triage_agent as fta
print("\nVERIFY  tools registered with the agent:", [t.name for t in fta.FraudTriageAgent().all_tools])
print("        (trace span table for the TXN-0234209 explain run is in fraud_triage_agent.ipynb)")

CHECK  objects read by the UC functions (parsed from their definitions)
  get_transaction_risk   reads ['fe_bar_lakebase.fraud.txn_features', 'fe_bar_lakebase.fraud.txn_risk_current']
                         comment: DEPRECATED 2026-10-21 (FGAC hardening Item 5): reads ungoverned fe_bar...
  explain_top_signals    reads ['fe_bar_lakebase.fraud.customer_risk_profile', 'fe_bar_lakebase.fraud.txn_features', 'fe_bar_lakebase.fraud.txn_risk_current']
                         comment: DEPRECATED 2026-10-21 (FGAC hardening Item 5): reads ungoverned fe_bar...
  get_customer_history   reads ['fe_bar_lakebase.cases.fraud_cases', 'fe_bar_lakebase.fraud.customer_risk_profile', 'fe_bar_lakebase.fraud.txn_features', 'fe_bar_lakebase.fraud.txn_risk_current', 'workshop_guy_catalog.fe_bar_a0.fact_prior_fraud_cases_silver']
                         comment: DEPRECATED 2026-10-21 (FGAC hardening Item 5): reads ungoverned fe_bar...

CHECK  UC path
  UC current_user: alex.owen@databricks.com | is_account_

/local_disk0/.ephemeral_nfs/envs/pythonEnv-937d2042-54d2-4ace-aa4c-9dfe43a4bd0c/lib/python3.12/site-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
2026/10/23 23:15:43 WARNING mlflow.utils.autologging_utils: MLflow langchain autologging is known to be compatible with 1.0.0 <= langchain, but the installed version is 0.3.30. If you encounter errors during autologging, try upgrading / downgrading langchain to a compatible version, or try upgrading MLflow.



VERIFY  tools registered with the agent: ['get_transaction_risk', 'explain_top_signals', 'get_customer_history', 'create_fraud_case']
        (trace span table for the TXN-0234209 explain run is in fraud_triage_agent.ipynb)


In [0]:
print("PRE-FIX CHECK (same session, before the fix): as fraud_triage_agent, INSERT with created_by")
print("  'fraud_triage_copilot:someone_else@example.com' SUCCEEDED (case 71cb5cac-0b11-4b16-8e13-3fd6a73ac200, rolled back).")
c = pg(); cur = c.cursor()
print("\nFIX  column + policies now in place:")
cur.execute("SELECT table_name, is_nullable, column_default FROM information_schema.columns WHERE table_schema='cases' AND column_name='db_inserted_by' ORDER BY 1")
for r in cur.fetchall(): print("  db_inserted_by on", r)
cur.execute("SELECT tablename, policyname, with_check FROM pg_policies WHERE schemaname='cases' AND cmd='INSERT' ORDER BY 1")
for r in cur.fetchall(): print("  policy", r[0], r[1], "WITH CHECK", r[2])
for t in ["fraud_cases", "case_actions"]:
    print(f"  backfill {t}: rows={one(cur, f'SELECT count(*) FROM cases.{t}')[0]}, null db_inserted_by={one(cur, f'SELECT count(*) FROM cases.{t} WHERE db_inserted_by IS NULL')[0]}")
txn7 = one(cur, """SELECT txn_id FROM fraud.txn_features WHERE amount <= 10000 AND txn_id <> 'TXN-0234209'
                   AND txn_id NOT IN (SELECT txn_id FROM cases.fraud_cases WHERE status <> 'closed') ORDER BY txn_id LIMIT 1""")[0]
c.close()
print("\nVERIFY 1: agent INSERT that sets db_inserted_by to a different value")
c = pg(autocommit=False); cur = c.cursor(); cur.execute("SET ROLE fraud_triage_agent")
try:
    cur.execute("""INSERT INTO cases.fraud_cases (txn_id, recommended_action, assigned_queue, notes, created_by, db_inserted_by)
                   VALUES (%s,'review','analyst_review','item7 spoof','fraud_triage_copilot:agent','someone_else@example.com')""", (txn7,))
    print("  UNEXPECTED: accepted")
except Exception as e:
    print(f"  REJECTED: {type(e).__name__}: {str(e).strip()}")
c.rollback(); c.close()
print("VERIFY 2: normal agent INSERT through LakebaseCases(pg_role='fraud_triage_agent')")
lc = LakebaseCases(pg_role="fraud_triage_agent")
cid = lc.create_fraud_case(txn7, "review", "analyst_review", "item7 normal insert", "fraud_triage_copilot:analyst@example.com")
lc.close()
admin = LakebaseCases(); case = admin.get_case(cid)
print(f"  case {cid}: created_by='{case['created_by']}' (agent-supplied label), db_inserted_by='{case['db_inserted_by']}' (connecting login)")
admin.update_case_status(cid, "closed", "fgac_hardening_item7_cleanup"); admin.close()
print("  cleanup: case closed")
print("\nHonest claim: the database records which identity connected; the analyst label is supplied by the agent.")

PRE-FIX CHECK (same session, before the fix): as fraud_triage_agent, INSERT with created_by
  'fraud_triage_copilot:someone_else@example.com' SUCCEEDED (case 71cb5cac-0b11-4b16-8e13-3fd6a73ac200, rolled back).

FIX  column + policies now in place:
  db_inserted_by on ('case_actions', 'NO', 'SESSION_USER')
  db_inserted_by on ('fraud_cases', 'NO', 'SESSION_USER')
  policy case_actions fraud_triage_agent_insert_actions WITH CHECK ((actor ~~ 'fraud_triage_copilot%'::text) AND (db_inserted_by = SESSION_USER) AND (EXISTS ( SELECT 1
   FROM cases.fraud_cases fc
  WHERE (fc.case_id = case_actions.case_id))))
  policy fraud_cases fraud_triage_agent_insert_cases WITH CHECK ((created_by ~~ 'fraud_triage_copilot%'::text) AND (db_inserted_by = SESSION_USER))
  backfill fraud_cases: rows=225, null db_inserted_by=0
  backfill case_actions: rows=450, null db_inserted_by=0

VERIFY 1: agent INSERT that sets db_inserted_by to a different value
  REJECTED: InsufficientPrivilege: new row violates row-leve

In [0]:
print("UC workshop_guy_catalog.fe_bar_a0.risk_score_calibration (test set, run d89beaca8f81447780da60ed2ea7c811, split 2026-09-21 15:20:18)")
cal = spark.table("workshop_guy_catalog.fe_bar_a0.risk_score_calibration").orderBy("bucket_order").toPandas()
print(cal[["bucket", "lower_bound", "upper_bound", "row_count", "fraud_count", "observed_precision", "confidence"]].to_string(index=False))
print("total test rows:", int(cal.row_count.sum()))
print("rule: high if observed_precision >= 0.70, medium if >= 0.30, else low")
st = w.postgres.get_synced_table(name="synced_tables/fe_bar_lakebase.fraud.risk_score_calibration")
print("\nsynced table fe_bar_lakebase.fraud.risk_score_calibration:", st.status.detailed_state.value, "| pipeline", st.status.pipeline_id)
c = pg(); cur = c.cursor(); cur.execute("SET ROLE fraud_triage_agent")
print("fraud_gov.risk_score_calibration as fraud_triage_agent: rows =", one(cur, "SELECT count(*) FROM fraud_gov.risk_score_calibration")[0])
c.close()
res = json.loads(fta.get_transaction_risk.invoke({"txn_id": "TXN-0234209"}))
print("\nget_transaction_risk('TXN-0234209') [" + res["_implementation"] + "] reads", res["_objects_read"])
for k, v in res["rows"][0].items(): print(f"  {k:<24} {v}")
import mlflow
runs = mlflow.search_runs(experiment_names=["/Users/alex.owen@databricks.com/fe_bar_fraud_triage_agent"], filter_string="tags.hardening_item = '8'")
r0 = runs.iloc[0]
print("\nMLflow run (calibration params):", r0.run_id, "| confidence_high_min_precision =", r0["params.confidence_high_min_precision"], "| medium =", r0["params.confidence_medium_min_precision"], "| block_band:", r0["params.bucket_block_band"])

UC workshop_guy_catalog.fe_bar_a0.risk_score_calibration (test set, run d89beaca8f81447780da60ed2ea7c811, split 2026-09-21 15:20:18)
     bucket  lower_bound  upper_bound  row_count  fraud_count  observed_precision confidence
  decile_01     0.000000     0.120317       9293           47            0.005058        low
  decile_02     0.120317     0.136336       9293           52            0.005596        low
  decile_03     0.136336     0.146831       9292           43            0.004628        low
  decile_04     0.146831     0.156282       9293           60            0.006456        low
  decile_05     0.156282     0.166525       9292           56            0.006027        low
  decile_06     0.166525     0.178564       9293           45            0.004842        low
  decile_07     0.178564     0.193492       9293           51            0.005488        low
  decile_08     0.193492     0.215899       9292           51            0.005489        low
  decile_09     0.215899     0

2026/10/23 23:15:54 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.



get_transaction_risk('TXN-0234209') [agent_python:postgres:fraud_triage_agent] reads ['fraud_gov.txn_risk_current', 'fraud_gov.txn_features', 'fraud_gov.risk_score_calibration']
  txn_id                   TXN-0234209
  customer_id              CUST-08669
  txn_ts                   2026-06-27T13:08:48+00:00
  amount                   323.96
  risk_score               0.9844108757126732
  recommended_action       block
  model_version            1
  scored_at                2026-10-21T02:36:46.583767+00:00
  confidence               medium
  observed_precision       0.5
  calibration_bucket       block_band
  calibration_bucket_rows  288
  routed_queue             auto_block
  confidence_display       medium (observed precision 0.50 in block_band, n=288)

MLflow run (calibration params): b78f0a3ba3f04ab6b970d1da4744bb9e | confidence_high_min_precision = 0.7 | medium = 0.3 | block_band: [0.950267,1.000000) n=288 fraud=144 precision=0.5000 conf=medium


Trace(trace_id='tr-f0a5b84ea23e76cf7b047ec04b532792')

In [0]:
import pii_role_sync as prs
plan = prs.build_plan()
print("DRY RUN: UC group 'pii' -> Postgres role fraud_pii (no changes made)")
prs.print_table(plan)
print("\nApply for own identity only: the plan action for alex.owen@databricks.com is",
      [p["action"] for p in plan if p["identity"] == ME], "-> no-op, so nothing to apply. No roles created for anyone else.")
print("Production path: scheduled job running pii_role_sync.py --apply (see logs/agent_governance_2026-10-21.md).")

DRY RUN: UC group 'pii' -> Postgres role fraud_pii (no changes made)
identity                 | kind | pg_role_exists | member_of_fraud_pii | action
-------------------------+------+----------------+---------------------+-------
alex.owen@databricks.com | USER | True           | True                | none  

Apply for own identity only: the plan action for alex.owen@databricks.com is ['none'] -> no-op, so nothing to apply. No roles created for anyone else.
Production path: scheduled job running pii_role_sync.py --apply (see logs/agent_governance_2026-10-21.md).


In [0]:
print("fgac_hardening: check/verify only. No fix applied, job not re-run; Item 1 and Item 7 test cases were closed in their cells.")

fgac_hardening: check/verify only. No fix applied, job not re-run; Item 1 and Item 7 test cases were closed in their cells.
